# Fig 9 — Gate-level early-abort cycle saving

Solid curve = **exact** expected saving (statevector, no sampling), extended to large $n$; open markers = noiseless AerSimulator means (validation, $n\leq12$). The saving rises monotonically toward a $\approx43\%$ asymptote — it does **not** saturate near 35%.

Canonical figure: `reproduce/circuits/standalone_aer_validation_empirical.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from rodeo_ness import circuits as C
rcParams.update({"font.family":"serif","font.serif":["DejaVu Serif"],
                 "mathtext.fontset":"dejavuserif","axes.linewidth":0.8})
BLUE,GREEN = "#0072B2","#009E73"; H=0.5
# A few noiseless Aer reps (n<=12) to VALIDATE the exact curve. Canonical figure
# via reproduce/circuits/standalone_aer_validation_empirical.py.
N_REPEATS, SHOTS = 4, 15000
recs = C.run_convergence_sweep(h=H, n_values=range(1,13), shots=SHOTS, seed=0, n_repeats=N_REPEATS)
n   = np.array([r["n"] for r in recs], float)
avg = np.array([r["avg_executed_cycles"] for r in recs], float)
sav = np.array([r["cycle_saving"] for r in recs], float)*100

In [ ]:
# exact expected curve (no sampling), extended past the Aer range, + asymptote
NN = np.arange(1,51); ex = np.array([C.expected_executed_cycles(H,int(k)) for k in NN])
ex_sav = (1-ex/NN)*100
slope = (C.expected_executed_cycles(H,400)-C.expected_executed_cycles(H,200))/200.0
asymp = (1-slope)*100
fig,(axA,axB)=plt.subplots(1,2,figsize=(11.5,4.6))
axA.plot(NN,NN,"--",color="0.6",lw=1.3,label="static (all $n$ cycles)")
axA.plot(NN,ex,"-",color=BLUE,lw=2,label="expected (exact)")
axA.plot(n,avg,"o",color=BLUE,ms=6,mfc="white",mec=BLUE,label=f"AerSimulator ({N_REPEATS} reps)")
axA.set_xlabel("number of scheduled cycles $n$"); axA.set_ylabel("executed cycle bodies / shot")
axA.set_title("(a) early abort: executed vs scheduled"); axA.legend(fontsize=9,loc="upper left"); axA.grid(alpha=0.2)
axB.plot(NN,ex_sav,"-",color=GREEN,lw=2,label="expected (exact)")
axB.plot(n,sav,"o",color=GREEN,ms=6,mfc="white",mec=GREEN,label=f"AerSimulator ({N_REPEATS} reps)")
axB.axhline(asymp,ls="--",color="0.4",lw=1.3,label=f"$n\\to\\infty$ asymptote $\\approx{asymp:.0f}\\%$")
axB.set_xlabel("number of scheduled cycles $n$"); axB.set_ylabel("mean cycle saving (%)")
axB.set_title("(b) saving keeps rising, slowly approaching its asymptote")
axB.set_ylim(0,asymp+4); axB.legend(fontsize=9,loc="lower right"); axB.grid(alpha=0.2)
fig.tight_layout(); fig.savefig("aer_cycle_saving.pdf",bbox_inches="tight"); plt.show()
print(f"saving asymptote ~ {asymp:.1f}% (does NOT saturate at 35%)")